In [1]:
import imageio.v3 as iio
import numpy as np
from tqdm import tqdm
from pathlib import Path
import imageio_ffmpeg as ffmpeg
import itertools
from src.data.preprocess.respiratory_extraction import resp_extraction
from hydra import initialize, compose
from omegaconf import OmegaConf

/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.0.4) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
video_dir = Path("/tf/00_data/#_2021_Sleep_Video/")
video_files = list(video_dir.glob("A*/*.mp4"))

In [3]:
with initialize(config_path="config", job_name="resp_extraction"):
    cfg = compose(config_name="preprocess/respiratory").preprocess
    print(OmegaConf.to_yaml(cfg))

log:
  dir: /tf/01_code/mylittlecodes/SleepVST_baseline/logs
  name: respiratory_extraction
video:
  input_path: /tf/00_data/#_2021_Sleep_Video/
  file_list: /tf/01_code/mylittlecodes/SleepVST_baseline/config/preprocess/file_list.txt
output:
  dir: /tf/01_code/mylittlecodes/SleepVST_baseline/data/respiratory
  save_frames: true
  save_signals: true
  save_magnified_frames: true
video_fps: 30
magnification:
  mag_factor: 50
  freq_range:
  - 0.2
  - 0.3
  attenuate: true
  sigma: 5
temporal_filter:
  type: difference_of_iir
  rl: 0.4
  rh: 0.05
num_workers: 1
skip_existing: false
batch_size: 1



<ipython-input-3-9b04a3a2f323>:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="config", job_name="resp_extraction"):


In [4]:
def loadVideoStream(reader_iter, num_frames):
    '''
    Load RGB video frames from an active iterator
    - reader_iter: active imageio iterator
    - num_frames: number of frames to read
    Returns:
        numpy array of frames or None if stream ended
    '''
    frames = []
    try:
        for _ in tqdm(range(num_frames)):
            frame = next(reader_iter)
            frames.append(frame)
    except StopIteration:
        # Stream ended
        if len(frames) == 0:
            return None
        # Return partial frames if any
        return np.array(frames)
    
    return np.array(frames)

def get_last_epoch_fps(json_path):
    """
    Args:
        json_path (str)
    Returns:
        int: 마지막 에포크 번호
    """
    import json
    import datetime

    with open(json_path, 'r') as f:
        ann = json.load(f)
        fps = ann["Video_Info"][0]["Frame_Rate"]
        record_id = ann["Case_Info"]["Case_Number"]
        start_time = datetime.datetime.strptime(ann['Video_Info'][0]['Start'], "%Y/%m/%d %H:%M:%S.%f")
        end_time = datetime.datetime.strptime(ann['Video_Info'][0]['End'], "%Y/%m/%d %H:%M:%S.%f")
        total_epoch = (end_time - start_time).seconds // 30
        return total_epoch, fps, record_id

In [7]:
def difference_of_iir(delta, rl, rh):
    lowpass_1 = delta[0].copy()
    lowpass_2 = lowpass_1.copy()
    out = np.zeros(delta.shape, dtype=delta.dtype)
    for i in range(1, delta.shape[0]):
        lowpass_1 = (1-rh)*lowpass_1 + rh*delta[i]
        lowpass_2 = (1-rl)*lowpass_2 + rl*delta[i]
        out[i] = lowpass_1 - lowpass_2
    return out
    

def get_temporal_filter(cfg):
    """
    Create temporal filter function based on config.

    Args:
        cfg: Configuration object with temporal filter settings

    Returns:
        Temporal filter function
    """
    filter_type = cfg.temporal_filter.get('type', 'difference_of_iir')

    if filter_type == 'difference_of_iir':
        rl = cfg.temporal_filter.get('rl', 0.4)
        rh = cfg.temporal_filter.get('rh', 0.05)

        def temporal_filter(delta, fl, fh):
            return difference_of_iir(delta, rl, rh)

        return temporal_filter
    else:
        raise ValueError(f"Unknown temporal filter type: {filter_type}")

In [8]:
video_path = str(video_files[0])
video_name = video_files[0].stem
video_output_dir = Path("/tf/01_code/mylittlecodes/SleepVST_baseline/outputs/preprocessed_data") / video_name
video_output_dir.mkdir(parents=True, exist_ok=True)

num_epochs, fps, record_id = get_last_epoch_fps(video_path.replace('_video_01.mp4', '_annotation.json'))

reader = iio.imiter(video_path)
reader_iter = iter(reader)
frames_per_epoch = int(30 * fps)

for i in tqdm(range(num_epochs), mininterval=10, desc=f"Processing epochs for {video_name}"):
    # Load frames for current epoch from stream
    epoch_video = loadVideoStream(reader_iter, frames_per_epoch)
    
    epoch_output_dir = video_output_dir / f'epoch_{i+1}'
    epoch_output_dir.mkdir(parents=True, exist_ok=True)
    
    max_point = resp_extraction(
            video=epoch_video,
            fps=fps,
            mag_factor=cfg.magnification.mag_factor,
            freq_range=cfg.magnification.freq_range,
            attenuate=cfg.magnification.attenuate,
            sigma=cfg.magnification.sigma,
            temporal_filter=get_temporal_filter(cfg),
            save_dir=epoch_output_dir,
            epoch_idx=i+1
        )
    

Processing epochs for A2019-EM-01-0057_video_01:   0%|          | 0/895 [00:00<?, ?it/s]

100%|██████████| 150/150 [00:00<00:00, 338.40it/s]


YIQ video: Done.


FFT video: 100%|##########| 150/150 [00:00<00:00, 289.43it/s]


Pyramid height:  6
Number of Filters:  8


...

YIQ video: Done.


FFT video: 100%|##########| 150/150 [00:00<00:00, 272.39it/s]


Pyramid height:  6
Number of Filters:  8


Bandpassing:  83%|########3 | 5/6 [00:52<00:10, 10.42s/it]
Processing epochs for A2019-EM-01-0057_video_01:   1%|          | 5/895 [06:39<19:45:57, 79.95s/it]


KeyboardInterrupt: 

In [ ]:
def loadVideo(file_dir, start_frame=0, end_frame=None):
    '''
    Load RGB video using imageio (uint8)
    - file_dir: video file path (ex> end with ".mp4")
    '''
    reader = iio.imiter(file_dir)
    orig_vid = []
    for i, im in tqdm(enumerate(itertools.islice(reader, start_frame, end_frame)), 
                      ascii=True, 
                      desc="Load Video"):
        orig_vid.append(im)
        
    return np.array(orig_vid)

In [8]:
video = loadVideo(str(video_files[0]), start_frame=30 * 500 * 5, end_frame=30 * 501 * 5)

Load Video: 45650it [01:11, 641.09it/s] 


KeyboardInterrupt: 

In [33]:
video.shape

(900, 480, 640, 3)

In [22]:
frames = iio.imiter(
        str(video_files[0]),
        plugin='FFMPEG',
        params=['-ss', "30", "-t", "30"]
    )
type(frames)

generator

In [23]:
for frame in frames:
    print(frame.shape)
    break

TypeError: _open() got an unexpected keyword argument 'params'

In [ ]:
reader = ffmpeg.read_frames(str(video_files[0]), start_time=30, end_time=60)

TypeError: _open() got an unexpected keyword argument 'params'